<a href="https://colab.research.google.com/github/Aniket0610/GenAI/blob/main/Railways_GenAI_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q -U langgraph langchain-core langchain-openai nest_asyncio chromadb sentence-transformers openai pandas

In [4]:
import os
import re
import nest_asyncio
import pandas as pd
from typing import Annotated, Optional, TypedDict

from google.colab import userdata

from langchain_core.tools import StructuredTool
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

nest_asyncio.apply()

In [5]:
# api key — set this secret in Colab (key icon on the left sidebar)
openrouter_key = userdata.get("OPENROUTER_API_KEY")

os.environ["OPENAI_API_KEY"] = openrouter_key

# ChatOpenAI pointed at OpenRouter's OpenAI-compatible endpoint
# "openrouter/free" auto-routes to an available free model that supports tool calling
llm = ChatOpenAI(
    model="openrouter/free",
    temperature=0,
    api_key=openrouter_key,
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "Indian Railways Gen-AI Assistant"
    }
)

print("openrouter key loaded:", openrouter_key is not None)

openrouter key loaded: True


In [6]:
# fake/sample railway booking database
# in real life this is the IRCTC database, here just a csv/dataframe

bookings = []

bookings.append({
    "ticket_id": "TKT-IR-10234",
    "txn_id": "TXN-88452167",
    "pnr": "2451098765",
    "name": "Aniket Lad",
    "train_no": "12951",
    "train_name": "Mumbai Rajdhani",
    "from": "NDLS",
    "to": "BCT",
    "date": "2026-09-25",
    "class": "3A",
    "coach": "B4",
    "berth": "22 Side Lower",
    "status": "CNF",
    "fare": 2145,
    "payment_mode": "UPI",
    "refund": "N/A"
})

bookings.append({
    "ticket_id": "TKT-IR-10235",
    "txn_id": "TXN-88452201",
    "pnr": "2451098766",
    "name": "Yash Parab",
    "train_no": "12301",
    "train_name": "Howrah Rajdhani",
    "from": "HWH",
    "to": "NDLS",
    "date": "2026-09-27",
    "class": "2A",
    "coach": "A1",
    "berth": "05 Lower",
    "status": "RAC 2",
    "fare": 3560,
    "payment_mode": "Credit Card",
    "refund": "N/A"
})

bookings.append({
    "ticket_id": "TKT-IR-10236",
    "txn_id": "TXN-88452299",
    "pnr": "2451098767",
    "name": "Atharv Thorat",
    "train_no": "22691",
    "train_name": "Rajdhani Express",
    "from": "SBC",
    "to": "NDLS",
    "date": "2026-08-30",
    "class": "SL",
    "coach": "S6",
    "berth": "-",
    "status": "CANCELLED",
    "fare": 890,
    "payment_mode": "Net Banking",
    "refund": "Rs 801 credited on 2026-09-02"
})

bookings.append({
    "ticket_id": "TKT-IR-10237",
    "txn_id": "TXN-88452355",
    "pnr": "2451098768",
    "name": "Shubham Terse",
    "train_no": "12626",
    "train_name": "Kerala Express",
    "from": "NDLS",
    "to": "TVC",
    "date": "2026-10-05",
    "class": "SL",
    "coach": "S3",
    "berth": "44 Upper",
    "status": "CNF",
    "fare": 745,
    "payment_mode": "UPI",
    "refund": "N/A"
})

railway_db = pd.DataFrame(bookings)
railway_db.to_csv("railway_bookings.csv", index=False)
railway_db

,ticket_id,txn_id,pnr,name,train_no,train_name,from,to,date,class,coach,berth,status,fare,payment_mode,refund
0,TKT-IR-10234,TXN-88452167,2451098765,Aniket Lad,12951,Mumbai Rajdhani,NDLS,BCT,2026-09-25,3A,B4,22 Side Lower,CNF,2145,UPI,N/A
1,TKT-IR-10235,TXN-88452201,2451098766,Yash Parab,12301,Howrah Rajdhani,HWH,NDLS,2026-09-27,2A,A1,05 Lower,RAC 2,3560,Credit Card,N/A
2,TKT-IR-10236,TXN-88452299,2451098767,Atharv Thorat,22691,Rajdhani Express,SBC,NDLS,2026-08-30,SL,S6,-,CANCELLED,890,Net Banking,Rs 801 credited on 2026-09-02
3,TKT-IR-10237,TXN-88452355,2451098768,Shubham Terse,12626,Kerala Express,NDLS,TVC,2026-10-05,SL,S3,44 Upper,CNF,745,UPI,N/A


In [7]:
# both ticket_id and transaction_id are mandatory here
# if either is missing or they dont match the same row, we deny access

def authenticate(ticket_id, txn_id):
    if not ticket_id or not txn_id:
        return None

    row = railway_db[(railway_db["ticket_id"] == ticket_id) & (railway_db["txn_id"] == txn_id)]

    if row.empty:
        return None

    return row.iloc[0]


def get_pnr_status(ticket_id: str, transaction_id: str) -> str:
    # returns pnr and current status, needs both ids
    rec = authenticate(ticket_id, transaction_id)
    if rec is None:
        return "Access denied. Ticket ID and Transaction ID are both required and must match."

    result = "PNR " + rec["pnr"] + " | Train " + rec["train_no"] + " (" + rec["train_name"] + ")"
    result = result + " | " + rec["from"] + " to " + rec["to"] + " on " + rec["date"]
    result = result + " | Status: " + rec["status"]
    return result


def get_seat_details(ticket_id: str, transaction_id: str) -> str:
    # returns class, coach, berth
    rec = authenticate(ticket_id, transaction_id)
    if rec is None:
        return "Access denied. Ticket ID and Transaction ID are both required and must match."

    return "Class: " + rec["class"] + " | Coach: " + rec["coach"] + " | Berth: " + rec["berth"]


def get_refund_status(ticket_id: str, transaction_id: str) -> str:
    # returns fare paid and refund status
    rec = authenticate(ticket_id, transaction_id)
    if rec is None:
        return "Access denied. Ticket ID and Transaction ID are both required and must match."

    return "Fare Paid: Rs " + str(rec["fare"]) + " | Refund Status: " + rec["refund"]


def get_fare_details(ticket_id: str, transaction_id: str) -> str:
    # returns fare and payment mode
    rec = authenticate(ticket_id, transaction_id)
    if rec is None:
        return "Access denied. Ticket ID and Transaction ID are both required and must match."

    return "Fare Paid: Rs " + str(rec["fare"]) + " | Payment Mode: " + rec["payment_mode"]


# quick test
print(get_pnr_status("TKT-IR-10234", "TXN-88452167"))
print(get_refund_status("TKT-IR-10236", "TXN-88452299"))
print(get_pnr_status("TKT-IR-10234", ""))
print(get_pnr_status("TKT-IR-10234", "TXN-WRONG"))

PNR 2451098765 | Train 12951 (Mumbai Rajdhani) | NDLS to BCT on 2026-09-25 | Status: CNF
Fare Paid: Rs 890 | Refund Status: Rs 801 credited on 2026-09-02
Access denied. Ticket ID and Transaction ID are both required and must match.
Access denied. Ticket ID and Transaction ID are both required and must match.


In [8]:
# small RAG setup for general railway questions
# these questions dont need ticket id / transaction id since they are not personal

from sentence_transformers import SentenceTransformer
import chromadb

faq_docs = [
    "Cancellation charges for AC First or Executive class is Rs 240 flat if cancelled more than 48 hours before departure.",
    "Cancellation charges for AC 3 tier or AC Chair Car is Rs 180 flat if cancelled more than 48 hours before departure.",
    "Cancellation charges for Sleeper class is Rs 120 flat if cancelled more than 48 hours before departure.",
    "If cancelled between 48 hours and 12 hours before departure, 25 percent of fare is deducted.",
    "If cancelled within 12 hours to 4 hours before departure, 50 percent of fare is deducted.",
    "Refund for cancelled confirmed e-tickets is credited to original payment source in 5 to 7 working days.",
    "Waitlisted tickets that do not get confirmed are cancelled automatically and refund is processed without any action from the user.",
    "Luggage allowance is 40kg for Sleeper, 40kg for AC 3 tier, 50kg for AC 2 tier and 70kg for AC First.",
    "Excess luggage beyond free allowance is charged 1.5 times the normal rate at the parcel office.",
    "Tatkal tickets generally cannot be cancelled for a refund except in a few special cases like train cancellation.",
    "Senior citizens can request lower berth preference at the time of booking, subject to availability."
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="railway_faq")
except Exception:
    pass

faq_collection = chroma_client.create_collection(name="railway_faq")

faq_ids = []
for i in range(len(faq_docs)):
    faq_ids.append("faq_" + str(i))

faq_embeddings = embedder.encode(faq_docs).tolist()
faq_collection.add(documents=faq_docs, embeddings=faq_embeddings, ids=faq_ids)

print("faq docs indexed:", len(faq_docs))


def search_railway_policy(query: str) -> str:
    # general policy search, no ids needed
    query_embedding = embedder.encode([query]).tolist()
    results = faq_collection.query(query_embeddings=query_embedding, n_results=3)

    passages = results["documents"][0]
    output = ""
    for p in passages:
        output = output + "- " + p + "\n"

    return output


print(search_railway_policy("cancel my ac 3 tier ticket"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

faq docs indexed: 11
- Cancellation charges for AC 3 tier or AC Chair Car is Rs 180 flat if cancelled more than 48 hours before departure.
- Cancellation charges for AC First or Executive class is Rs 240 flat if cancelled more than 48 hours before departure.
- Waitlisted tickets that do not get confirmed are cancelled automatically and refund is processed without any action from the user.



In [9]:
# wrapping the functions above as tools the agent can call

pnr_tool = StructuredTool.from_function(
    func=get_pnr_status,
    name="get_pnr_status",
    description="Get PNR and booking status. Needs ticket_id and transaction_id, both mandatory."
)

seat_tool = StructuredTool.from_function(
    func=get_seat_details,
    name="get_seat_details",
    description="Get coach, berth, class of a ticket. Needs ticket_id and transaction_id, both mandatory."
)

refund_tool = StructuredTool.from_function(
    func=get_refund_status,
    name="get_refund_status",
    description="Get refund status of a ticket. Needs ticket_id and transaction_id, both mandatory."
)

fare_tool = StructuredTool.from_function(
    func=get_fare_details,
    name="get_fare_details",
    description="Get fare and payment mode of a ticket. Needs ticket_id and transaction_id, both mandatory."
)

faq_tool = StructuredTool.from_function(
    func=search_railway_policy,
    name="search_railway_policy",
    description="Answer general railway policy questions like cancellation or luggage rules. No ids needed."
)

railway_tools = [pnr_tool, seat_tool, refund_tool, fare_tool, faq_tool]

for t in railway_tools:
    print(t.name)

get_pnr_status
get_seat_details
get_refund_status
get_fare_details
search_railway_policy


In [10]:
# graph state
class RailwayState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    ticket_id: Optional[str]
    transaction_id: Optional[str]
    needs_personal: bool
    next_node: str


ticket_pattern = re.compile(r"TKT-IR-\d+", re.IGNORECASE)
txn_pattern = re.compile(r"TXN-\d+", re.IGNORECASE)

# keywords that signal a personal/account-specific query (needs both ids)
# anything not matching these is treated as a general policy question
personal_keywords = [
    "pnr", "seat", "coach", "berth", "refund", "fare paid", "payment mode",
    "my ticket", "my booking", "my status", "status of my", "my refund",
    "my fare", "my seat", "my coach", "my pnr", "cancel my", "my train"
]


def classify_intent(text: str) -> bool:
    # returns True if this looks like a personal query needing ids
    text_lower = text.lower()
    for kw in personal_keywords:
        if kw in text_lower:
            return True
    return False

In [11]:
# pulls ticket id / transaction id out of the user message
# in a real app this would come from the logged in session instead
# also classifies whether this looks like a personal query at all

def intake_node(state):
    last_msg = None
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage):
            last_msg = m
            break

    text = last_msg.content

    ticket_found = ticket_pattern.search(text)
    txn_found = txn_pattern.search(text)

    new_ticket_id = state.get("ticket_id")
    new_txn_id = state.get("transaction_id")

    if ticket_found:
        new_ticket_id = ticket_found.group(0).upper()

    if txn_found:
        new_txn_id = txn_found.group(0).upper()

    needs_personal = classify_intent(text)

    return {
        "ticket_id": new_ticket_id,
        "transaction_id": new_txn_id,
        "needs_personal": needs_personal
    }

In [12]:
# ids are only mandatory when the query actually needs personal data
# general policy questions skip straight to the agent, no ids required

def gatekeeper_node(state):
    needs_personal = state.get("needs_personal", False)

    if not needs_personal:
        return {"next_node": "agent"}

    has_ticket = state.get("ticket_id") is not None
    has_txn = state.get("transaction_id") is not None

    if has_ticket and has_txn:
        return {"next_node": "agent"}
    else:
        return {"next_node": "request_ids"}


def route_after_gatekeeper(state):
    return state["next_node"]

In [13]:
# if ids are missing, just ask the user for them, dont go to the agent

def request_ids_node(state):
    missing = []

    if state.get("ticket_id") is None:
        missing.append("Ticket ID")

    if state.get("transaction_id") is None:
        missing.append("Transaction ID")

    missing_str = " and ".join(missing)

    reply = "I need your " + missing_str + " to look up this booking. "
    reply = reply + "Both Ticket ID and Transaction ID are mandatory for any personal query. "
    reply = reply + "General policy questions dont need any id."

    return {"messages": [AIMessage(content=reply)], "next_node": END}

In [14]:
# hand off to a react agent with the tools
# ids may or may not be present depending on whether this is a personal query

def agent_node(state):
    ticket_id = state.get("ticket_id")
    transaction_id = state.get("transaction_id")

    if ticket_id and transaction_id:
        system_text = "You are the Indian Railways Gen-AI assistant. "
        system_text = system_text + "The verified Ticket ID is " + ticket_id + " and verified Transaction ID is " + transaction_id + ". "
        system_text = system_text + "Use these two ids whenever you call a tool that needs them. "
    else:
        system_text = "You are the Indian Railways Gen-AI assistant. "
        system_text = system_text + "No verified Ticket ID or Transaction ID is available right now. "
        system_text = system_text + "Only answer general policy questions using search_railway_policy. "
        system_text = system_text + "Do not call get_pnr_status, get_seat_details, get_refund_status, or get_fare_details, since no ids are available."

    system_text = system_text + " Never make up booking data, only use what the tools return. "
    system_text = system_text + "For general policy questions use search_railway_policy."

    system_msg = SystemMessage(content=system_text)

    react_agent = create_react_agent(llm, railway_tools)
    result = react_agent.invoke({"messages": [system_msg] + state["messages"]})

    final_msg = result["messages"][-1]

    return {"messages": [final_msg], "next_node": END}

In [15]:
# build the graph

builder = StateGraph(RailwayState)

builder.add_node("intake", intake_node)
builder.add_node("gatekeeper", gatekeeper_node)
builder.add_node("request_ids", request_ids_node)
builder.add_node("agent", agent_node)

builder.add_edge(START, "intake")
builder.add_edge("intake", "gatekeeper")

builder.add_conditional_edges(
    "gatekeeper",
    route_after_gatekeeper,
    {
        "agent": "agent",
        "request_ids": "request_ids"
    }
)

builder.add_edge("agent", END)
builder.add_edge("request_ids", END)

railway_app = builder.compile()
print("graph compiled")

graph compiled


In [16]:
# helper to test the assistant

def ask_railway_assistant(user_query):
    print("USER:", user_query)

    inputs = {
        "messages": [HumanMessage(content=user_query)],
        "ticket_id": None,
        "transaction_id": None,
        "needs_personal": False
    }

    result = railway_app.invoke(inputs)
    final_reply = result["messages"][-1].content

    print("ASSISTANT:", final_reply)
    print("-" * 60)

In [17]:
# test 1 - both ids given, pnr status
ask_railway_assistant("What is the PNR status of my ticket? Ticket ID: TKT-IR-10234, Transaction ID: TXN-88452167")

USER: What is the PNR status of my ticket? Ticket ID: TKT-IR-10234, Transaction ID: TXN-88452167


/tmp/ipykernel_7744/2656278464.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_agent = create_react_agent(llm, railway_tools)


ASSISTANT: Based on the PNR status for your ticket, here are the details:

**PNR Status: 2451098765**
- **Train:** 12951 (Mumbai Rajdhani)
- **Route:** NDLS to BCT (New Delhi to Bandra Terminus)
- **Date:** September 25, 2026
- **Status:** CNF (Confirmed)

Your ticket is confirmed and you have a reserved seat on the Mumbai Rajdhani Express from New Delhi to Bandra Terminus.
------------------------------------------------------------


In [18]:
# test 2 - no ids given, should be refused
ask_railway_assistant("Can you check my refund status please?")

USER: Can you check my refund status please?
ASSISTANT: I need your Ticket ID and Transaction ID to look up this booking. Both Ticket ID and Transaction ID are mandatory for any personal query. General policy questions dont need any id.
------------------------------------------------------------


In [19]:
# test 3 - both ids given, refund status on a cancelled ticket
ask_railway_assistant("Ticket ID TKT-IR-10236, Transaction ID TXN-88452299, what is my refund status?")

USER: Ticket ID TKT-IR-10236, Transaction ID TXN-88452299, what is my refund status?


/tmp/ipykernel_7744/2656278464.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_agent = create_react_agent(llm, railway_tools)


ASSISTANT: Your refund status for Ticket ID TKT-IR-10236 is as follows:

- **Fare Paid:** Rs 890
- **Refund Status:** Rs 801 has been credited to your account on 2026-09-02.

If you have any more questions or need further assistance, feel free to ask!
------------------------------------------------------------


In [20]:
# test 4 - general question, no ids needed
ask_railway_assistant("What is the luggage allowance for AC 2 tier passengers?")

USER: What is the luggage allowance for AC 2 tier passengers?


/tmp/ipykernel_7744/2656278464.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_agent = create_react_agent(llm, railway_tools)


ASSISTANT: The luggage allowance for AC 2 tier passengers is 50kg.
------------------------------------------------------------
